In [188]:
from bs4 import BeautifulSoup
import pandas as pd
import re
import requests
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver.common.by import By
import requests
import ast
import plotly.graph_objects as go
import plotly.express as px
from tqdm import tqdm
from datetime import datetime

In [264]:

df_datasets = pd.read_excel('./Palabras Claves.xlsx')


In [65]:

#df_datasets = df_datasets[df_datasets['Words'].str.contains('Analytical modeling|Food insecurity')]
#df_datasets.reset_index(drop=True, inplace=True)
#df_datasets

In [173]:
driver = webdriver.Chrome()
# Abre la página web
url = 'https://grants.nih.gov/funding/searchguide/index.html#/'
driver.get(url)


In [174]:
checkbox = driver.find_element(By.NAME , 'nosisCheck')
if checkbox.is_selected():
    checkbox.click()

In [175]:
driver.find_element(By.CLASS_NAME, "advancedSearch.mb-3").click()

In [176]:
search_input = driver.find_element(By.ID, 'react-select-4-input')  
search_input.clear()
search_input.send_keys('Research and Development')
search_input.send_keys(Keys.ENTER)



In [177]:
search_input = driver.find_element(By.ID, 'react-select-11-input')  
search_input.clear()
search_input.send_keys('Research Projects')
search_input.send_keys(Keys.ENTER)

In [178]:
search_input = driver.find_element(By.ID, 'react-select-6-input')  
search_input.clear()
search_input.send_keys('New')
search_input.send_keys(Keys.ENTER)

In [179]:

r_list = ['RC2', 'RM1', 'R01', 'R03', 'R13', 'R15', 'R16', 'R18', 'R21',
'R24', 'R25', 'R34', 'R35', 'R36', 'R44', 'R50',
'R61', 'R41/R42', 'R43/R44', 'R21/R33', 'R61/R33','K99/R00', 'R33','R00']

'''
r_list = ['K99/R00','R00']
'''
search_input = driver.find_element(By.ID, 'react-select-8-input')  
for i in r_list:    
    search_input.send_keys(i)
    time.sleep(1)
    search_input.send_keys(Keys.ENTER)
    time.sleep(1)


In [180]:

search_button = driver.find_element(By.XPATH, "//button[text()='Search']") 
search_button.click()


In [286]:

def find_number_results(word):

    actions = ActionChains(driver) 
    search_input = driver.find_element(By.ID, 'searchQuery')

    actions.move_to_element(search_input).perform() # Hacer clic en el botón 
    time.sleep(1)
    
    search_input.clear()
    search_input.send_keys(f'"{word}"')

    time.sleep(7)
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    texto = soup.select_one('div.col-lg-6.col-12.d-flex.align-items-center.mb-3').text
    match = re.search(r'of (\d+)', texto)
    if match:
        valor = match.group(1)
    elif texto != '':
        valor = texto.split(":")[1].strip()
    else:
        valor = '0' 
    return word, valor

In [287]:

def recolecta_informacion(soup):
    lista = []
    for row in soup.select('table.table-striped.border.table-bordered.responsive-table tbody tr'):
        renglon = []
        renglon.append(row.select('td')[0].text)
        renglon.append(row.select('td')[1].text)
        renglon.append(row.select('td')[1].select_one('a')['href'])
        renglon.append(row.select('td')[2].text)
        renglon.append(row.select('td')[3].text)
        renglon.append(row.select('td')[4].text)
        renglon.append(row.select('td')[5].text)
        renglon.append(row.select('td')[6].text)
        lista.append(renglon)
    return lista

In [288]:
def if_canContinue(driver):
    try:
        driver.find_element(By.CLASS_NAME, 'next.disabled')
        return False
    except:
        return True

In [289]:

def next_pages(driver):
    intentos = 0
    actions = ActionChains(driver) 

    while True:
        #print(intentos)
        try:
            
            soup_nextPages = driver.find_element(By.CLASS_NAME, 'next')
            actions.move_to_element(soup_nextPages).perform() # Hacer clic en el botón 
            time.sleep(1)
            soup_nextPages.click()

            break
        except:
            intentos += 1

            if intentos == 3:
                break

In [306]:

def colect_infoGrant(driver,number):
    tittles = ['Title', 'Notice Number','url','Issuing Organization', 'Participating Organization', 'Release Date','Expiration Date','Activity Codes']
    acumulador = pd.DataFrame()


    try:
        actions = ActionChains(driver) 

        first_page_button = driver.find_element(By.XPATH, "//li/a[text()='1']") # Hacer clic en el enlace de la primera página 

        actions.move_to_element(first_page_button).perform() # Hacer clic en el botón 
        time.sleep(1)
        first_page_button.click()
    except:
        pass

    ##Valida Cambio de pagina
    while True:

        if number<20:
            time.sleep(7)
            soup = BeautifulSoup (driver.page_source, 'html.parser')
            lista = recolecta_informacion(soup)
            df_description = pd.DataFrame(lista, columns=tittles)
            acumulador = pd.concat([acumulador, df_description], ignore_index=True)
            break
        elif if_canContinue(driver):
            time.sleep(7)
            soup = BeautifulSoup (driver.page_source, 'html.parser')
            lista = recolecta_informacion(soup)
            df_description = pd.DataFrame(lista, columns=tittles)
            acumulador = pd.concat([acumulador, df_description], ignore_index=True)

            next_pages(driver)
        else:
            time.sleep(7)
            soup = BeautifulSoup (driver.page_source, 'html.parser')
            lista = recolecta_informacion(soup)
            df_description = pd.DataFrame(lista, columns=tittles)
            acumulador = pd.concat([acumulador, df_description], ignore_index=True)
            break

    return acumulador



In [314]:


results = []

total_words = len(df_datasets)  

df_grants = pd.DataFrame()

for i in tqdm(range(total_words), desc="Procesando palabras"):

    temporal = find_number_results(df_datasets['Words'][i])
    results.append([temporal[0], temporal[1]])

    if int(temporal[1]) > 0:
        df_temporal = colect_infoGrant(driver,int(temporal[1]))
        df_temporal['word'] = df_datasets['Words'][i]
        df_grants = pd.concat([df_grants, df_temporal], ignore_index=True)
        

    #print(df_datasets['Words'][i] , int(temporal[1]), len(df_temporal))
#driver.quit()

df_resultado = pd.DataFrame(results, columns=['words', 'Results'])

print("Proceso completado. Resultados almacenados en df_resultado.")
df_grants.to_excel('description_NIH.xlsx')
df_resultado.to_excel('./Resultados NIH.xlsx')



Procesando palabras: 100%|██████████| 35/35 [17:35<00:00, 30.16s/it]

Proceso completado. Resultados almacenados en df_resultado.


In [316]:
iterr = ["Education and bioinformatics", "Education and Biostatistics","Education and Molecular profiling", "Education and climate change","Education and Computational biology",
        "Education or bioinformatics", "Education or Biostatistics","Education or Molecular profiling", "Education or climate change","Education or Computational biology",
        '"Education" "bioinformatics"', '"Education" "Biostatistics"', '"Education" "Molecular profiling"','"Education" "climate change"','"Education" "Computational biology"']

In [317]:
results = []

total_words = len(df_datasets)  

df_grants = pd.DataFrame()

for i in iterr:

    temporal = find_number_results(i)
    results.append([temporal[0], temporal[1]])

    if int(temporal[1]) > 0:
        df_temporal = colect_infoGrant(driver,int(temporal[1]))
        df_temporal['word'] = i
        df_grants = pd.concat([df_grants, df_temporal], ignore_index=True)
        

    #print(df_datasets['Words'][i] , int(temporal[1]), len(df_temporal))
#driver.quit()

df_resultado = pd.DataFrame(results, columns=['words', 'Results'])

print("Proceso completado. Resultados almacenados en df_resultado.")
df_grants.to_excel('description_NIH combitation.xlsx')
df_resultado.to_excel('./Resultados NIH combitation.xlsx')


Proceso completado. Resultados almacenados en df_resultado.


In [268]:
next_pages(driver)

0


#################################################################

Segunda Parte

#################################################################

### Recorrido por cada una de las URLS

In [5]:
pd_listado_url = pd.read_excel('./listado.xlsx')

In [60]:
for i in range(len(pd_listado_url.head(10))):

    print(pd_listado_url['url'][i], ' - ', pd_listado_url['title'][i])
    
    response = requests.get(pd_listado_url['url'][i])
    html_content = response.text

    html = BeautifulSoup (html_content, 'html.parser')

    nombre_archivo = str('./funding/{}.txt').format(str(pd_listado_url['url'][i]).split('/')[-1:][0].replace('.html',''))
    with open(nombre_archivo, 'w') as archivo:
        archivo.write(str(html))
    time.sleep(3)
    

https://grants.nih.gov/grants/guide/notice-files/NOT-OD-22-135.html  -  Notice of Special Interest (NOSI): Stimulating Research to Understand and Address Hunger, Food and Nutrition Insecurity


Notice of Special Interest (NOSI): Stimulating Research to Understand and Address Hunger, Food and Nutrition Insecurity
https://grants.nih.gov/grants/guide/rfa-files/RFA-MD-24-005.html  -  Elucidating Mechanisms Associated with HIV Related Co-Morbidities in Populations Experiencing Health Disparities (R01 - Clinical Trials Not Allowed)
https://grants.nih.gov/grants/guide/pa-files/PAS-24-163.html  -  Priority HIV/AIDS Research within the Mission of NIDDK (R01 Clinical Trial Optional)
https://grants.nih.gov/grants/guide/rfa-files/RFA-DK-25-001.html  -  Addressing the Impact of Syndemics on the Health of People with HIV and Diseases and Conditions within the Missions of NIDDK and NHLBI (R01 Clinical Trial Optional)
https://grants.nih.gov/grants/guide/rfa-files/RFA-DA-25-048.html  -  Seeking Product